# Independent ESN wipe research — Mac + CUDA workstation workflow

Architecture: the task-specific ESN receives only $[q_t,\dot q_t]$ and runs closed-loop at 100 Hz. Frozen UnifoLM is a **training-time** teacher at 570 ms anchors, never a deployment input. Ridge behavior cloning is only initialization; MuJoCo task optimization uses black-box policy search.

**Mac-compatible stages:** capability audit, dataset packing, BC init, visited-bundle collection, full-pose IK, task-only MuJoCo optimization, cache validation, artifact loading.

**CUDA-only stage:** frozen `unitreerobotics/UnifoLM-VLA-Base` labeling with `allow_mock_fallback=False` and `unnorm_key=g1_wipe_table`. Mock / demonstration-proxy labels are forbidden in final results.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
MJCF = ROOT / 'unitree_mujoco/unitree_robots/g1/g1_29dof.xml'
RESULTS = ROOT / 'results/main_independent_esn'
FINAL = RESULTS / 'workstation_final'
RESULTS.mkdir(parents=True, exist_ok=True)
from wipe_esn_experiment import ESN, fit_bc_initializer, pack_episodes, rollout
from unifolm_teacher_pipeline import mac_capability_report, save_visited_bundle, validate_cache
mac_capability_report()

## 1. Mac capability decision

Apple Silicon runs MuJoCo and PyTorch Metal. Unitree's official UnifoLM release targets Python 3.10 / CUDA (FlashAttention2 preferred; SDPA fallback OK). Real action-head labeling is therefore a **CUDA-worker** stage. Collection, ESN fitting, MuJoCo optimization, cache validation, and 23-D EE→29-D IK all run on Mac or the lab box.

In [ ]:
ds = load_dataset('unitreerobotics/G1_Dex1_Wipe_Table')
TRAIN_EPS = [0, 1, 2, 3]
VAL_EPS = [4, 5, 6]          # inside training range 0–159 only
HELDOUT_EPS = list(range(160, 200))  # frozen held-out; never used for selection
episodes = pack_episodes(ds['train'], TRAIN_EPS + VAL_EPS + HELDOUT_EPS[:3])
esn = ESN(n=116, seed=0)
fit_bc_initializer(esn, {ep: episodes[ep] for ep in TRAIN_EPS})

## 2. Collect teacher observations from states the ESN actually visits

Each 570 ms anchor stores rendered RGB, visited 29-D joint state/velocity, and validated 23-D EE proprioception. Re-run collection after every accepted policy update (DAgger-style), then label that new bundle on the frozen CUDA teacher.

In [ ]:
bundle = RESULTS / 'visited_ep0_seed0.npz'
if not bundle.exists():
    result = rollout(esn, episodes[0], MJCF, capture_anchors=True, teacher_weight=0.0)
    save_visited_bundle(result['captures'], bundle, episode=0, seed=0,
                        policy_id='bc_initializer_seed0', dagger_round=0)
bundle_meta = json.loads(bundle.with_suffix('.json').read_text())
bundle_meta

## 3. Frozen UnifoLM + full-pose IK labeling (CUDA-only)

Run in the official UnifoLM Python-3.10/CUDA environment (do **not** force FlashAttention into the Mac `.venv`):

```bash
export HF_HOME=/raid/data/aihimekpen/hf_cache   # lab box
export PYTHONPATH=/path/to/research:/path/to/unifolm-vla/src
python notebooks/unifolm_teacher_pipeline.py label \
  results/main_independent_esn/visited_ep0_seed0.npz \
  results/main_independent_esn/teacher_ep0_seed0.npz \
  --mjcf unitree_mujoco/unitree_robots/g1/g1_29dof.xml \
  --model unitreerobotics/UnifoLM-VLA-Base --unnorm-key g1_wipe_table
```

The labeler sets `allow_mock_fallback=False`, sends RGB + language + 23-D visited-state proprioception to frozen UnifoLM, selects **chunk step 0** of the 25×23 action chunk (immediate next EE action), converts through position-and-orientation Jacobian IK, and writes 29-D joint anchors.

**Full workstation reproduction** (DAgger + optimizers + all-40 held-out + ablations):

```bash
PYTHONPATH=research:unifolm-vla/src:research/notebooks \
  python research/notebooks/run_workstation_final_experiment.py
```

In [ ]:
teacher_cache = RESULTS / 'teacher_ep0_seed0.npz'
teacher_ready = teacher_cache.exists()
if teacher_ready:
    visited = np.load(bundle)
    teacher_status = validate_cache(
        teacher_cache,
        expected_anchors=bundle_meta['anchors'],
        expected_times=visited['time_s'],
    )
else:
    teacher_status = {
        'ready': False,
        'blocker': 'Official frozen-UnifoLM cache must be produced on CUDA worker; mock/proxy labels are forbidden.',
    }
teacher_status

## 4. IK acceptance test

Full-pose IK solves a damped 6-D Jacobian problem (position + orientation). The saved held-out test uses simultaneous ±1 cm position and ±5° orientation perturbations and requires p95 error below 5 mm and 2°.

In [ ]:
ik_validation = json.loads((RESULTS / 'ik_validation.json').read_text())
assert ik_validation['pass']
ik_validation

## 5. Literature-grounded task objective

Success requires grasp, ≥0.768 m wipe path (5th percentile of all 200 demonstrations), ≥90% geometric table contact, ≥90% bounded tabletop coverage, no joint-limit violation, and no >5 cm mocap cloth jump. Read `results/main_independent_esn/TASK_OBJECTIVE.md`. Geometric contact is not a calibrated force claim.

In [ ]:
print((RESULTS / 'TASK_OBJECTIVE.md').read_text())

## 6. Mac-compatible task-only optimizer smoke benchmark

If the real teacher cache is absent, the Mac benchmark remains **task-only** and is not the final teacher-guided result.

```bash
python run_full_mac_experiment.py --budget 8 --seeds 0,1,2
```

In [ ]:
mac_bench = RESULTS / 'mac_optimizer_benchmark.json'
if mac_bench.exists():
    benchmark = json.loads(mac_bench.read_text())
    summary = {}
    for method in benchmark['methods']:
        rows = [r for r in benchmark['rows'] if r['method'] == method]
        summary[method] = {
            'heldout_L_task_mean': float(np.mean([r['heldout_L_task'] for r in rows])),
            'grasps': sum(r['grasp_success'] for r in rows),
            'successes': sum(r['task_success'] for r in rows),
            'trials': len(rows),
        }
    print('teacher_status:', benchmark.get('teacher_status'))
    summary
else:
    {'note': 'Mac benchmark not present on this machine'}

## 7. Load completed workstation final artifacts (preferred)

After `run_workstation_final_experiment.py` finishes on the CUDA lab box, this cell loads the frozen results without re-running UnifoLM. Artifacts live under `results/main_independent_esn/workstation_final/` (symlinked to `/raid` on the DGX).

In [ ]:
required = [
    'run_config.json', 'environment_audit.json', 'teacher_cache_manifest.json',
    'dagger_history.json', 'optimizer_comparison.json', 'heldout_summary.json',
    'FINAL_REPORT.md',
]
final_ready = FINAL.is_dir() and all((FINAL / n).exists() for n in required)
if not final_ready:
    {
        'ready': False,
        'blocker': 'Run research/notebooks/run_workstation_final_experiment.py on the CUDA workstation.',
        'missing': [n for n in required if not (FINAL / n).exists()],
    }
else:
    run_config = json.loads((FINAL / 'run_config.json').read_text())
    heldout = json.loads((FINAL / 'heldout_summary.json').read_text())
    opt = json.loads((FINAL / 'optimizer_comparison.json').read_text())
    manifest = json.loads((FINAL / 'teacher_cache_manifest.json').read_text())
    assert manifest.get('mock_any') is False, 'mock teacher caches are forbidden'
    print((FINAL / 'FINAL_REPORT.md').read_text()[:2500])
    {
        'selected_method': opt.get('selected_method'),
        'selection_criterion': opt.get('selection_criterion'),
        'heldout_success_rate': heldout.get('success_rate'),
        'heldout_ci95': heldout.get('success_ci95'),
        'n_teacher_caches': len(manifest.get('caches', [])),
        'elapsed_s': run_config.get('elapsed_s'),
    }


In [ ]:
if final_ready:
    from IPython.display import Image, display
    for name in ('optimizer_curves.png', 'heldout_metrics.png'):
        p = FINAL / name
        if p.exists():
            display(Image(filename=str(p)))
    import pandas as pd
    sens = pd.read_csv(FINAL / 'threshold_sensitivity.csv')
    display(sens)
else:
    print('Final artifacts not available yet.')
